# DDM: estimation error against the number of trials (TINN, PyDDM, BayesFlow)

For each number of trials (50, 100, 500, 1,000 and 2,000), `N_DATASETS` datasets are simulated with
different true parameters and estimated by the three methods; the RMSE of each parameter is computed
over these datasets. The true parameters are the same at every number of trials, so only the amount of
data changes. All steps are done by `rmse_trials.py` in this folder.

Results go to `results/ddm_ntrials/`. Every step continues where it stopped when run again; keep the
number of workers the same between runs.

## 1. Settings

In [ ]:
TRIALS = [50, 100, 500, 1000, 2000]   # number of samples (RTs) per dataset
N_DATASETS = 20                       # datasets per number of samples (RMSE is over these)

RUN_PYDDM, RUN_TINN, RUN_BAYESFLOW = True, True, False   # the paper (Figure 13) shows TINN and PyDDM
PYDDM_WORKERS = 4
TINN_WORKERS = 3          # parallel TINN processes on the GPU
TINN_WARM, TINN_ITERS = 2000, 6000
TINN_DEVICE = "auto"      # "auto" = GPU if there is one
BF_SIM, BF_EPOCHS = 20000, 50

## 2. Results folder and helper

In [ ]:
import os, sys, subprocess

CODE = os.path.abspath(".")                 # this folder (2_parameter_estimation)
assert os.path.exists(os.path.join(CODE, "rmse_trials.py")), "open this notebook from 2_parameter_estimation/"
R = os.path.join(CODE, "results", "ddm_ntrials")
os.makedirs(R, exist_ok=True)
print("results folder:", R)

In [ ]:
def run(*args):
    """runs one step of rmse_trials.py and shows its output as it comes"""
    cmd = [sys.executable, "-u", "rmse_trials.py", *map(str, args), "--root", R,
           "--trials", *map(str, TRIALS)]
    p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line:
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError("failed (see the messages above)")

## 3. Simulated datasets

In [ ]:
run("make", "--n-datasets", N_DATASETS)

## 4. PyDDM

In [ ]:
if RUN_PYDDM:
    run("fit", "--method", "pyddm", "--workers", PYDDM_WORKERS)

## 5. TINN

The slowest step: a few minutes per dataset on a GPU, divided over the workers.

In [ ]:
if RUN_TINN:
    run("fit", "--method", "tinn", "--workers", TINN_WORKERS, "--warm", TINN_WARM, "--iters", TINN_ITERS,
        "--device", TINN_DEVICE)

## 6. BayesFlow (optional, not in the paper)

Trained separately for every number of trials, in its own process (`fit_bayesflow.py`). The training log is `log_bayesflow.txt` in the folder of each N.

In [ ]:
if RUN_BAYESFLOW:
    run("fit", "--method", "bayesflow", "--n-sim", BF_SIM, "--epochs", BF_EPOCHS)

## 7. Figure

Output: `rmse_trials.pdf` (black and white) and `rmse_trials.csv` in the results folder. Methods that have not finished are left out.

In [ ]:
run("plot")

from IPython.display import Image, display
display(Image(os.path.join(R, "rmse_trials.png"), width=1000))